# IDM-VTON — Virtual Try-On Server

This notebook hosts IDM-VTON as a Gradio API. After running all cells, you'll get a **public URL** that the kiosk edge function calls.

**GPU Required:** A100 recommended (T4 may OOM at 768×1024). Go to Runtime → Change runtime type → A100.

**How to use:**
1. Run all cells
2. Copy the `https://xxxxx.gradio.live` URL
3. Set it as `IDM_VTON_COLAB_URL` secret in Supabase

In [ ]:
#@title 1. Install Dependencies
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q diffusers==0.25.1 transformers accelerate safetensors
!pip install -q gradio huggingface_hub Pillow
!pip install -q bitsandbytes scipy

print('Dependencies installed!')

In [ ]:
#@title 2. Clone IDM-VTON repo & install
import os
if not os.path.exists('IDM-VTON'):
    !git clone https://github.com/yisol/IDM-VTON.git
os.chdir('IDM-VTON')
!pip install -q -r requirements.txt 2>/dev/null || true
print('Repo ready!')

In [ ]:
#@title 3. Download model weights
from huggingface_hub import snapshot_download

# Download IDM-VTON weights
snapshot_download(
    repo_id="yisol/IDM-VTON",
    local_dir="./checkpoints",
    ignore_patterns=["*.md", ".gitattributes"]
)
print('Model weights downloaded!')

In [ ]:
#@title 4. Load the model pipeline
import sys
sys.path.insert(0, '.')

import torch
from PIL import Image
from diffusers import AutoencoderKL, DDPMScheduler
from transformers import AutoTokenizer, CLIPTextModel, CLIPTextModelWithProjection, CLIPVisionModelWithProjection, CLIPImageProcessor
import io
import base64
import numpy as np

# Check GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

# Try loading the pipeline using the approach from the demo
try:
    from src.tryon_pipeline import StableDiffusionXLInpaintPipeline as TryonPipeline
    from src.unet_hacked_tryon import UNet2DConditionModel
    from src.unet_hacked_garmnet import UNet2DConditionModel as UNet2DConditionModel_ref
    from preprocess.humanparsing.run_parsing import Parsing
    from preprocess.openpose.run_openpose import OpenPose
    
    CKPT = './checkpoints'
    
    # Load UNets
    unet = UNet2DConditionModel.from_pretrained(
        CKPT, subfolder='unet', torch_dtype=torch.float16
    ).to(device)
    
    unet_encoder = UNet2DConditionModel_ref.from_pretrained(
        CKPT, subfolder='unet_encoder', torch_dtype=torch.float16
    ).to(device)
    
    # Load other components
    tokenizer_one = AutoTokenizer.from_pretrained(CKPT, subfolder='tokenizer')
    tokenizer_two = AutoTokenizer.from_pretrained(CKPT, subfolder='tokenizer_2')
    text_encoder_one = CLIPTextModel.from_pretrained(CKPT, subfolder='text_encoder', torch_dtype=torch.float16).to(device)
    text_encoder_two = CLIPTextModelWithProjection.from_pretrained(CKPT, subfolder='text_encoder_2', torch_dtype=torch.float16).to(device)
    image_encoder = CLIPVisionModelWithProjection.from_pretrained(CKPT, subfolder='image_encoder', torch_dtype=torch.float16).to(device)
    vae = AutoencoderKL.from_pretrained(CKPT, subfolder='vae', torch_dtype=torch.float16).to(device)
    
    noise_scheduler = DDPMScheduler.from_pretrained(CKPT, subfolder='scheduler')
    ip_ckpt = os.path.join(CKPT, 'ip_adapter.bin')
    
    pipe = TryonPipeline.from_pretrained(
        CKPT,
        unet=unet,
        vae=vae,
        feature_extractor=CLIPImageProcessor(),
        text_encoder=text_encoder_one,
        text_encoder_2=text_encoder_two,
        tokenizer=tokenizer_one,
        tokenizer_2=tokenizer_two,
        scheduler=noise_scheduler,
        image_encoder=image_encoder,
        torch_dtype=torch.float16,
    ).to(device)
    
    pipe.unet_encoder = unet_encoder
    
    # Load parsing and openpose
    parsing_model = Parsing(0)
    openpose_model = OpenPose(0)
    
    MODEL_LOADED = True
    print('IDM-VTON pipeline loaded successfully!')
except Exception as e:
    print(f'Error loading full pipeline: {e}')
    print('Will fall back to HuggingFace Spaces API')
    MODEL_LOADED = False

In [ ]:
#@title 5. Define try-on function
from PIL import Image
import io
import base64
import requests
import numpy as np

def load_image_from_url(url):
    """Download image from URL and return PIL Image."""
    response = requests.get(url, timeout=30)
    return Image.open(io.BytesIO(response.content)).convert('RGB')

def try_on_local(person_img, garment_img, description='clothing item', num_steps=30):
    """Run IDM-VTON inference locally."""
    # Resize to expected dimensions
    person_img = person_img.resize((768, 1024))
    garment_img = garment_img.resize((768, 1024))
    
    # Get parsing and pose
    keypoints = openpose_model(person_img.resize((384, 512)))
    parse_result, _ = parsing_model(person_img.resize((384, 512)))
    
    # Create mask from parsing
    parse_array = np.array(parse_result)
    # Upper body labels: 4=upper-clothes, 7=left-arm, 8=right-arm
    mask = (parse_array == 4).astype(np.float32)
    mask = Image.fromarray((mask * 255).astype(np.uint8)).resize((768, 1024))
    
    # Create pose image
    pose_img = person_img  # simplified, real impl uses openpose visualization
    
    with torch.no_grad(), torch.cuda.amp.autocast():
        result = pipe(
            prompt=description,
            negative_prompt='low quality, bad anatomy, deformed',
            image=person_img,
            mask_image=mask,
            image_garm=garment_img,
            image_vton=[pose_img],
            image_ori=person_img,
            num_inference_steps=num_steps,
            guidance_scale=2.0,
            seed=42,
            height=1024,
            width=768,
        ).images[0]
    
    return result

def try_on_via_spaces(person_img, garment_img, description='clothing item'):
    """Fallback: use HuggingFace Spaces demo API."""
    from gradio_client import Client, handle_file
    import tempfile
    
    # Save images to temp files
    with tempfile.NamedTemporaryFile(suffix='.jpg', delete=False) as f:
        person_img.save(f, 'JPEG')
        person_path = f.name
    with tempfile.NamedTemporaryFile(suffix='.jpg', delete=False) as f:
        garment_img.save(f, 'JPEG')
        garment_path = f.name
    
    client = Client('yisol/IDM-VTON')
    result = client.predict(
        dict={"background": handle_file(person_path), "layers": [], "composite": None},
        garm_img=handle_file(garment_path),
        garment_des=description,
        is_checked=True,
        is_checked_crop=False,
        denoise_steps=30,
        seed=42,
        api_name="/tryon"
    )
    
    # Result is a tuple, first element is the output image path
    return Image.open(result[0])

def virtual_try_on(person_url, garment_url, description='clothing item'):
    """Main entry point — tries local first, falls back to Spaces."""
    # Download images
    person_img = load_image_from_url(person_url)
    garment_img = load_image_from_url(garment_url)
    
    if MODEL_LOADED:
        try:
            result = try_on_local(person_img, garment_img, description)
            # Convert to base64
            buf = io.BytesIO()
            result.save(buf, format='JPEG', quality=90)
            b64 = base64.b64encode(buf.getvalue()).decode()
            return {"success": True, "image_base64": b64, "method": "local"}
        except Exception as e:
            print(f'Local inference failed: {e}, trying Spaces...')
    
    try:
        result = try_on_via_spaces(person_img, garment_img, description)
        buf = io.BytesIO()
        result.save(buf, format='JPEG', quality=90)
        b64 = base64.b64encode(buf.getvalue()).decode()
        return {"success": True, "image_base64": b64, "method": "spaces_api"}
    except Exception as e:
        return {"success": False, "error": str(e)}

print('Try-on function defined!')

In [ ]:
#@title 6. Launch Gradio API Server
import gradio as gr
import json

def gradio_tryon(person_url, garment_url, description):
    """Gradio wrapper — accepts URLs, returns JSON string."""
    result = virtual_try_on(person_url, garment_url, description or 'clothing item')
    return json.dumps(result)

demo = gr.Interface(
    fn=gradio_tryon,
    inputs=[
        gr.Textbox(label='Person Image URL', placeholder='https://...signed-url...'),
        gr.Textbox(label='Garment Image URL', placeholder='https://...signed-url...'),
        gr.Textbox(label='Description', value='clothing item'),
    ],
    outputs=gr.Textbox(label='Result JSON'),
    title='IDM-VTON API Server',
    description='Virtual Try-On API. Send person + garment image URLs.',
    api_name='tryon',
)

print('\n' + '='*60)
print('STARTING IDM-VTON SERVER...')
print('Copy the public URL below and set it as')
print('IDM_VTON_COLAB_URL in your Supabase secrets.')
print('='*60 + '\n')

demo.launch(share=True, quiet=False)